# Debug Existing RAG Pipeline Modules

Use this notebook to test the modules implemented so far:

1. `DocumentLoader`
2. `VietnameseTextPreprocessor`
3. `TextSplitter`
4. `EmbeddingModel`
5. `VectorStore`

Run the cells from top to bottom. The embedding and ChromaDB cells require the full dependencies from `requirements.txt`.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('c:/Users/ADMIN/Documents/RAG_document_read')

## 1. Create A Small Vietnamese TXT Sample

In [3]:
sample_path = PROJECT_ROOT / "data" / "samples" / "debug_vietnamese.txt"
sample_path.parent.mkdir(parents=True, exist_ok=True)

sample_path.write_text(
    "Sinh viên phải hoàn\nthành tối thiểu 65 % tổng số tín chỉ để được đăng ký thực tập.\n\n"
    "Sinh viên cần có điểm trung bình tích lũy đạt yêu cầu của khoa.\n\n"
    "Những câu hỏi không có trong tài liệu cần được hệ thống từ chối trả lời.",
    encoding="utf-8",
)

sample_path

WindowsPath('c:/Users/ADMIN/Documents/RAG_document_read/data/samples/debug_vietnamese.txt')

## 2. Test DocumentLoader

In [4]:
from app.document_loader import DocumentLoader

loader = DocumentLoader()
documents = loader.load(str(sample_path), sample_path.name, "debug-doc")
documents

[{'text': 'Sinh viên phải hoàn\nthành tối thiểu 65 % tổng số tín chỉ để được đăng ký thực tập.\n\nSinh viên cần có điểm trung bình tích lũy đạt yêu cầu của khoa.\n\nNhững câu hỏi không có trong tài liệu cần được hệ thống từ chối trả lời.',
  'metadata': {'file_id': 'debug-doc',
   'file_name': 'debug_vietnamese.txt',
   'page': 1}}]

## 3. Test VietnameseTextPreprocessor

In [5]:
from app.text_preprocessor import VietnameseTextPreprocessor

preprocessor = VietnameseTextPreprocessor()
processed_documents = []

for document in documents:
    processed_documents.append(
        {
            "text": preprocessor.clean(document["text"]),
            "metadata": document["metadata"],
        }
    )

processed_documents

[{'text': 'Sinh viên phải hoàn thành tối thiểu 65% tổng số tín chỉ để được đăng ký thực tập.\n\nSinh viên cần có điểm trung bình tích lũy đạt yêu cầu của khoa.\n\nNhững câu hỏi không có trong tài liệu cần được hệ thống từ chối trả lời.',
  'metadata': {'file_id': 'debug-doc',
   'file_name': 'debug_vietnamese.txt',
   'page': 1}}]

## 4. Test TextSplitter

In [6]:
from app.text_splitter import TextSplitter

splitter = TextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split(processed_documents)

for chunk in chunks:
    print(chunk["chunk_id"])
    print(chunk["metadata"])
    print(chunk["text"])
    print("-" * 80)

debug-doc_page_1_chunk_0
{'file_id': 'debug-doc', 'file_name': 'debug_vietnamese.txt', 'page': 1, 'chunk_index': 0}
Sinh viên phải hoàn thành tối thiểu 65% tổng số tín chỉ để được đăng ký thực tập.
--------------------------------------------------------------------------------
debug-doc_page_1_chunk_1
{'file_id': 'debug-doc', 'file_name': 'debug_vietnamese.txt', 'page': 1, 'chunk_index': 1}
ợc đăng ký thực tập.

Sinh viên cần có điểm trung bình tích lũy đạt yêu cầu của khoa.
--------------------------------------------------------------------------------
debug-doc_page_1_chunk_2
{'file_id': 'debug-doc', 'file_name': 'debug_vietnamese.txt', 'page': 1, 'chunk_index': 2}
ạt yêu cầu của khoa.

Những câu hỏi không có trong tài liệu cần được hệ thống từ chối trả lời.
--------------------------------------------------------------------------------


## 5. Optional: Test EmbeddingModel

This downloads/loads the multilingual sentence-transformers model if it is not already cached.

In [7]:
from app.config import get_config
from app.embedding_model import EmbeddingModel

config = get_config()
embedder = EmbeddingModel(config.embedding_model)
embeddings = embedder.encode([chunk["text"] for chunk in chunks])

len(embeddings), len(embeddings[0]) if embeddings else 0

c:\Users\ADMIN\Documents\RAG_document_read\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ADMIN\Documents\RAG_document_read\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode o

(3, 384)

## 6. Optional: Test VectorStore

In [8]:
from app.vector_store import VectorStore

debug_store = VectorStore(PROJECT_ROOT / "vector_db" / "debug", "debug_document_qa")
debug_store.reset()
debug_store.add_chunks(chunks, embeddings)

query = preprocessor.clean_query("Sinh viên cần bao nhiêu tín chỉ để đăng ký thực tập?")
query_embedding = embedder.encode([query])[0]
results = debug_store.search(query_embedding, top_k=3)
results

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[{'chunk_id': 'debug-doc_page_1_chunk_0',
  'text': 'Sinh viên phải hoàn thành tối thiểu 65% tổng số tín chỉ để được đăng ký thực tập.',
  'metadata': {'chunk_index': 0,
   'file_id': 'debug-doc',
   'file_name': 'debug_vietnamese.txt',
   'page': 1},
  'score': 0.7180043217268522,
  'distance': 0.28199567827314775},
 {'chunk_id': 'debug-doc_page_1_chunk_1',
  'text': 'ợc đăng ký thực tập.\n\nSinh viên cần có điểm trung bình tích lũy đạt yêu cầu của khoa.',
  'metadata': {'chunk_index': 1,
   'file_id': 'debug-doc',
   'file_name': 'debug_vietnamese.txt',
   'page': 1},
  'score': 0.6745841834866423,
  'distance': 0.3254158165133577},
 {'chunk_id': 'debug-doc_page_1_chunk_2',
  'text': 'ạt yêu cầu của khoa.\n\nNhững câu hỏi không có trong tài liệu cần được hệ thống từ chối trả lời.',
  'metadata': {'chunk_index': 2,
   'file_id': 'debug-doc',
   'file_name': 'debug_vietnamese.txt',
   'page': 1},
  'score': 0.19302282977298157,
  'distance': 0.8069771702270184}]